In [1]:
!pip install redshift_connector


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.9/164.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.6 MB/s eta 0:00:00


In [5]:
!pip install redshift_connector -q

import redshift_connector

conn = redshift_connector.connect(
    host     = "redshift-cluster-1.c1sd9uwaalma.us-east-1.redshift.amazonaws.com",
    port     = 5439,
    database = "datalake",
    user     = "lucas",
    password = "Lucas@Datalake2026!"
)

cursor = conn.cursor()

cursor.execute("""
SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name IN (
      'dim_data',
      'fato_geracao',
      'fato_taxa_teif_teip',
      'dim_usina',
      'dim_subsistema'
  )
ORDER BY table_name, ordinal_position;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

['dim_data', 'data_sk', 'integer']
['dim_data', 'data', 'date']
['dim_data', 'ano', 'integer']
['dim_data', 'mes', 'integer']
['dim_data', 'dia_mes', 'integer']
['dim_data', 'trimestre', 'integer']
['dim_data', 'dia_semana', 'integer']
['dim_data', 'fim_semana', 'boolean']
['dim_subsistema', 'sk_subsistema', 'bigint']
['dim_subsistema', 'id_subsistema', 'character varying']
['dim_subsistema', 'nom_subsistema', 'character varying']
['dim_usina', 'sk_usina', 'bigint']
['dim_usina', 'ceg', 'character varying']
['dim_usina', 'ceg_core', 'character varying']
['dim_usina', 'nom_usina', 'character varying']
['dim_usina', 'id_ons', 'character varying']
['dim_usina', 'id_estado', 'character varying']
['dim_usina', 'nom_estado', 'character varying']
['dim_usina', 'nom_tipousina', 'character varying']
['dim_usina', 'nom_tipocombustivel', 'character varying']
['dim_usina', 'nom_modalidadeoperacao', 'character varying']
['dim_usina', 'potencia_autorizada_mw', 'double precision']
['dim_usina', 'pote

In [9]:

conn = redshift_connector.connect(
    host     = "redshift-cluster-1.c1sd9uwaalma.us-east-1.redshift.amazonaws.com",
    port     = 5439,
    database = "datalake",
    user     = "lucas",
    password = "Lucas@Datalake2026!"
)

cursor = conn.cursor()

cursor.execute("""
WITH geracao_base AS (
    SELECT
        g.sk_usina,
        g.sk_subsistema,
        d.ano,
        d.mes,
        SUM(g.val_energia_mwh) AS energia_mwh
    FROM "datalake"."public"."fato_geracao" AS g
    INNER JOIN "datalake"."public"."dim_data" AS d
        ON g.sk_data = d.data_sk
    GROUP BY
        g.sk_usina,
        g.sk_subsistema,
        d.ano,
        d.mes
),
teif_base AS (
    SELECT
        t.sk_usina,
        d.ano,
        d.mes,
        AVG(CASE
                WHEN UPPER(t.nom_taxa) LIKE '%TEIF%'
                THEN t.val_taxa
            END) AS teif_media,
        AVG(CASE
                WHEN UPPER(t.nom_taxa) LIKE '%TEIP%'
                THEN t.val_taxa
            END) AS teip_media
    FROM "datalake"."public"."fato_taxa_teif_teip" AS t
    INNER JOIN "datalake"."public"."dim_data" AS d
        ON t.sk_data = d.data_sk
    GROUP BY
        t.sk_usina,
        d.ano,
        d.mes
),
joined AS (
    SELECT
        u.nom_usina,
        u.nom_tipousina,
        u.id_estado,
        s.nom_subsistema,
        g.ano,
        g.mes,
        g.energia_mwh,
        u.potencia_efetiva_mw,
        t.teif_media,
        t.teip_media,
        SUM(g.energia_mwh) OVER (
            PARTITION BY g.sk_usina, g.ano
            ORDER BY g.mes
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS energia_acumulada_ano,
        AVG(g.energia_mwh) OVER (
            PARTITION BY g.sk_usina
            ORDER BY g.ano, g.mes
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS media_movel_3m,
        RANK() OVER (
            PARTITION BY s.nom_subsistema, g.ano, g.mes
            ORDER BY g.energia_mwh DESC
        ) AS ranking_subsistema
    FROM geracao_base AS g
    INNER JOIN "datalake"."public"."dim_usina" AS u
        ON g.sk_usina = u.sk_usina
    INNER JOIN "datalake"."public"."dim_subsistema" AS s
        ON g.sk_subsistema = s.sk_subsistema
    LEFT JOIN teif_base AS t
        ON  g.sk_usina = t.sk_usina
        AND g.ano      = t.ano
        AND g.mes      = t.mes
)
SELECT
    nom_usina,
    nom_tipousina,
    id_estado,
    nom_subsistema,
    ano,
    mes,
    ROUND(energia_mwh, 2)           AS energia_mwh,
    ROUND(energia_acumulada_ano, 2) AS energia_acumulada_ano,
    ROUND(media_movel_3m, 2)        AS media_movel_3m,
    ROUND(potencia_efetiva_mw, 2)   AS potencia_efetiva_mw,
    ROUND(teif_media, 4)            AS teif,
    ROUND(teip_media, 4)            AS teip,
    ranking_subsistema
FROM joined
WHERE ranking_subsistema <= 5
ORDER BY
    ano DESC,
    mes DESC,
    nom_subsistema,
    ranking_subsistema
LIMIT 50;
""")

rows = cursor.fetchall()
print(f"Conexão OK! Linhas retornadas: {len(rows)}")
for row in rows[:10]:
    print(row)

Conexão OK! Linhas retornadas: 50
['UHE XINGÓ', 'HIDROELÉTRICA', 'SE', 'Nordeste', 2010, 12, 844204.01, 8534694.95, 795327.8099999999, 3162.0, None, None, 1]
['UHE PAULO AFONSO IV', 'HIDROELÉTRICA', 'BA', 'Nordeste', 2010, 12, 518313.89, 6131799.279999999, 512310.57, 2462.4, None, None, 2]
['UHE LUIZ GONZAGA', 'HIDROELÉTRICA', 'PE', 'Nordeste', 2010, 12, 337911.41000000003, 3447212.55, 321545.76, 1479.6, None, None, 3]
['UHE SOBRADINHO', 'HIDROELÉTRICA', 'BA', 'Nordeste', 2010, 12, 156990.33000000002, 1868317.3699999999, 144488.41, 1050.3, None, None, 4]
['UHE PAULO AFONSO III', 'HIDROELÉTRICA', 'BA', 'Nordeste', 2010, 12, 100780.08, 609940.46, 79202.28, 794.2, None, None, 5]
['UHE TUCURUI', 'HIDROELÉTRICA', 'PA', 'Norte', 2010, 12, 892976.51, 18907916.490000002, 808481.96, 8535.0, None, None, 1]
['UHE CURUÁ-UNA', 'HIDROELÉTRICA', 'PA', 'Norte', 2010, 12, 8523.6, 124452.47, 9240.93, 42.8, None, None, 2]
['GERAMAR II (ANTIGA NOVA OLINDA)', 'TÉRMICA', 'MA', 'Norte', 2010, 12, 0.0, 1838.5